In [1]:
from pyspark.sql import SparkSession
from urllib.request import urlretrieve

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("adsP1")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.driver.memory", "8G")
    .config("spark.executor.memory", "8G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/24 23:52:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/08/24 23:52:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/08/24 23:52:48 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import hour, col
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.mllib.evaluation import RegressionMetrics

In [3]:
df_model = spark.read.parquet("/Users/tianhao/Desktop/adsP1/data/curate/tlc_data/first_clean/df_all.parquet")

In [4]:
df_model.printSchema()

root
 |-- day_of_week: string (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- feels_like: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- visibility: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- is_peak_hour: integer (nullable = true)
 |-- avg_feels_like_peak: double (nullable = true)
 |-- avg_precipitation_peak: double (nullable = true)
 |-- avg_wind_speed_peak: double (nullable = true)
 |-- avg_visibility_peak: double (nullable = true)
 |-- avg_duration_minutes_peak: double (nullable = true)
 |-- avg_tr

In [5]:
# Assembly feature
feature_columns = ['trip_distance', 'wind_speed', 'precipitation', 'duration_minutes', 'passenger_count']
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Linear regression model

lr = LinearRegression(featuresCol="features", labelCol="total_revenue", predictionCol="prediction", regParam=0.1)

pipeline = Pipeline(stages=[assembler, lr])

train_data, test_data = df_model.randomSplit([0.8, 0.2], seed=42)

# fit model 
model = pipeline.fit(train_data)

# prediction on test set
predictions = model.transform(test_data)

# evalute model
evaluator = RegressionEvaluator(labelCol="total_revenue", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
predictionAndLabels = predictions.select("prediction", "total_revenue").rdd.map(lambda r: (float(r[0]), float(r[1])))
metrics = RegressionMetrics(predictionAndLabels)


24/08/24 23:53:08 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/08/24 23:53:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
/opt/anaconda3/envs/ADS/lib/python3.8/site-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [6]:
print("Linear Regression MAE: ", metrics.meanAbsoluteError)
print("Linear Regression MSE: ", metrics.meanSquaredError)
print("Linear Regression RMSE: ", metrics.rootMeanSquaredError)
print("Linear Regression R-squared: ", metrics.r2)

Linear Regression MAE:  5.867568853209313
Linear Regression MSE:  97.62312831100864
Linear Regression RMSE:  9.880441706270457
Linear Regression R-squared:  0.8159616543569516


## Random Forest To Predict The Revenue

In [7]:
# Assembly feature
feature_columns = ['trip_distance', 'wind_speed', 'precipitation', 'duration_minutes', 'passenger_count']
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Random forest model
rf = RandomForestRegressor(featuresCol="features", labelCol="total_revenue", predictionCol="prediction")

# The assembler and the random forest model are connected in series to form a pipeline
pipeline = Pipeline(stages=[assembler, rf])

# Split data set into training set and test set

train_data, test_data = df_model.randomSplit([0.8, 0.2], seed=42)

# Sampling 10% from the sampling dataset 
sampled_train_data = train_data.sample(False, 0.1, seed=42)  # 以 10% 的比例抽样

# Fit the model to the training set after sampling

model = pipeline.fit(sampled_train_data)

# Make predictions on the test set

predictions = model.transform(test_data)


evaluator = RegressionEvaluator(labelCol="total_revenue", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)

# Use RegressionMetrics for further assessment

predictionAndLabels = predictions.select("prediction", "total_revenue").rdd.map(lambda r: (float(r[0]), float(r[1])))
metrics = RegressionMetrics(predictionAndLabels)

In [8]:
print("Random Forest with 10% Sampling RMSE:", rmse)
print("Random Forest with 10% Sampling MSE:", metrics.meanSquaredError)
print("Random Forest with 10% Sampling R-squared:", metrics.r2)
print("Random Forest with 10% Sampling MAE:", metrics.meanAbsoluteError)

Random Forest with 10% Sampling RMSE: 10.068850117248747


Random Forest with 10% Sampling MSE: 101.38174268362012
Random Forest with 10% Sampling R-squared: 0.8088759444128706
Random Forest with 10% Sampling MAE: 5.798333434532924
